In [12]:
from dotenv import load_dotenv
from pathlib import Path  
envPath = Path.cwd().parent.parent.joinpath("env").joinpath("dev.aws.env")

print(envPath)

if envPath.exists():
    load_dotenv(dotenv_path= envPath, override=True)
    print("env loaded")
else:
    print("Env file missing")

/Users/vj/Sites/python-practice-combined/env/dev.aws.env
env loaded


In [13]:
from botocore.exceptions import ClientError
import boto3
import os

endpoint_url = os.getenv("AWS_URL")
region = os.getenv("REGION")
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")

print(os.getenv("APP_NAME"), endpoint_url, region, aws_access_key_id, aws_secret_access_key)

AWS PRACTICE MAC http://localhost:4566 us-east-1 vijay vijay


In [15]:
ec2 = boto3.resource('ec2',
    endpoint_url=endpoint_url,
    region_name=region,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)

ec2

ec2.ServiceResource()

In [ ]:

# try:
#     print("Creating a new EC2 instance...")
#     instances = ec2.create_instances(
#         ImageId='ami-df5db2b6',  # LocalStack accepts any AMI ID
#         MinCount=1,
#         MaxCount=1,
#         InstanceType='t2.micro'
#     )
#     instance = instances[0]
#     instance.wait_until_exists()
#     print(f"✓ Successfully created EC2 instance with ID: {instance.id}")
#     print(f"Current State: {instance.state['Name']}")
# except Exception as e:
#     print(f"Error creating EC2 instance: {e}")

In [ ]:
instance_id = 'i-xxxxxx' 

try:
    print(f"Terminating EC2 instance {instance_id}...")
    instance = ec2.Instance(instance_id)
    instance.terminate()
    instance.wait_until_terminated()
    print(f"✓ Successfully terminated instance. Current State: {instance.state['Name']}")
except Exception as e:
    print(f"Error terminating EC2 instance: {e}")

In [22]:
# Create new instance
try:
     print("Create new EC2 Instance")
     ec2_instance = ec2.create_instances(
        ImageId='text_id-0',  # LocalStack accepts any AMI ID
        MinCount=1,
        MaxCount=1,
        InstanceType='t2.2xlarge'
    )

     print(ec2_instance)   
except ClientError  as ce:
    print(ce.response["Error"]["Code"]) 

Create new EC2 Instance
[ec2.Instance(id='i-32eca3869d16c9f22')]


In [ ]:
try:
    print("Listing all EC2 instances:")
    instances = ec2.instances.all()
    count = 0
    for instance in instances:
        count += 1
        print(f"- ID: {instance.id} | State: {instance.state['Name']} | Type: {instance.instance_type} | AMI: {instance.image_id}")
    if count == 0:
        print("No EC2 instances found.")
except Exception as e:
    print(f"Error listing EC2 instances: {e}")


Listing all EC2 instances:
- ID: i-a52aa95f9d8dcd602 | State: running | Type: t2.micro | AMI: ami-df5db2b6
- ID: i-904bf94b5ea35d51a | State: running | Type: t2.micro | AMI: text_id-0
- ID: i-1ebdf7e2bf417f9dc | State: running | Type: t2.micro | AMI: text_id-0
- ID: i-0bdd8ab186adae2fc | State: running | Type: t2.micro | AMI: text_id-0
- ID: i-32eca3869d16c9f22 | State: running | Type: t2.2xlarge | AMI: text_id-0


In [ ]:
# Terminate instance code
instance_id = 'i-xxxxxx'  # Replace with actual instance ID

try:
    print(f"Terminating EC2 instance {instance_id}...")
    instance = ec2.Instance(instance_id)
    instance.terminate()
    instance.wait_until_terminated()
    print(f"✓ Successfully terminated instance. Current State: {instance.state['Name']}")
except Exception as e:
    print(f"Error terminating EC2 instance: {e}")

In [ ]:
# Reset EC2 Security Group inbound (ingress) rules
group_id = 'sg-xxxxxx'  # Replace with your security group ID
security_group = ec2.SecurityGroup(group_id)

try:
    ingress_rules = security_group.ip_permissions
    if ingress_rules:
        print(f"Clearing {len(ingress_rules)} inbound rule(s)...")
        security_group.revoke_ingress(IpPermissions=ingress_rules)
        print("✓ All inbound permissions reset (cleared).")
    else:
        print("No inbound rules found to reset.")
except Exception as e:
    print(f"Error resetting permissions: {e}")